In [ ]:
# !pip install ninja --break-system-packages

In [ ]:
# !pip install --upgrade transformers --break-system-packages

In [17]:
# !pip install auto-round==0.14.2 --break-system-packages

In [ ]:
# !pip install git+https://github.com/sustcsonglin/flash-linear-attention.git --no-build-isolation --break-system-packages

In [ ]:
# !pip install compressed-tensors --break-system-packages

In [1]:
import os

import torch
from auto_round import AutoRound
from huggingface_hub import HfApi, create_repo, get_token, notebook_login
from transformers import AutoModelForImageTextToText, AutoProcessor


In [2]:
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [3]:
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")


PyTorch Version: 2.13.0+cu130
CUDA Available: True
CUDA Version: 13.0
GPU Name: NVIDIA H100 PCIe
VRAM: 79.2 GB


In [4]:
MODEL_ID = "numind/NuExtract3"

OUTPUT_BASE_DIR = "./NuExtract3"
LOCAL_PATH = "./local_model_NuExtract3"

In [5]:
!hf download $MODEL_ID --local-dir $LOCAL_PATH

Reconstructing (incomplete total...): |           |  0.00B /  0.00B            

Fetching 29 files:   0%|                                | 0/29 [00:00<?, ?it/s]

Fetching 29 files: 100%|██████████████████████| 29/29 [00:00<00:00, 136.09it/s]
Download complete: :                                       |  0.00B            
Reconstruction complete: |                        |  0.00B /  0.00B            ✓ Downloaded
  path: /workspace/local_model_NuExtract3
Download complete: :                                       |  0.00B            
Reconstruction complete: |                        |  0.00B /  0.00B            


In [6]:
import os

from safetensors import safe_open

for file in os.listdir(LOCAL_PATH):
    if file.endswith(".safetensors"):
        path = os.path.join(LOCAL_PATH, file)
        print(f"\nChecking {file}")

        with safe_open(path, framework="pt") as f:
            keys = list(f.keys())

            mtp_keys = [k for k in keys if "mtp" in k.lower()]
            for k in mtp_keys:
                print(k)


Checking model.safetensors

Checking model_mtp.safetensors
mtp.fc.weight
mtp.layers.0.input_layernorm.weight
mtp.layers.0.mlp.down_proj.weight
mtp.layers.0.mlp.gate_proj.weight
mtp.layers.0.mlp.up_proj.weight
mtp.layers.0.post_attention_layernorm.weight
mtp.layers.0.self_attn.k_norm.weight
mtp.layers.0.self_attn.k_proj.weight
mtp.layers.0.self_attn.o_proj.weight
mtp.layers.0.self_attn.q_norm.weight
mtp.layers.0.self_attn.q_proj.weight
mtp.layers.0.self_attn.v_proj.weight
mtp.norm.weight
mtp.pre_fc_norm_embedding.weight
mtp.pre_fc_norm_hidden.weight


In [7]:
model = AutoModelForImageTextToText.from_pretrained(
    LOCAL_PATH, 
    dtype=torch.bfloat16, 
    device_map="auto"
)
processor = AutoProcessor.from_pretrained(LOCAL_PATH)

tokenizer = processor.tokenizer


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

In [8]:
model

Qwen3_5ForConditionalGeneration(
  (model): Qwen3_5Model(
    (visual): Qwen3_5VisionModel(
      (patch_embed): Qwen3_5VisionPatchEmbed(
        (proj): Conv3d(3, 1024, kernel_size=(2, 16, 16), stride=(2, 16, 16))
      )
      (pos_embed): Embedding(2304, 1024)
      (rotary_pos_emb): Qwen3_5VisionRotaryEmbedding()
      (blocks): ModuleList(
        (0-23): 24 x Qwen3_5VisionBlock(
          (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True, bias=True)
          (norm2): LayerNorm((1024,), eps=1e-06, elementwise_affine=True, bias=True)
          (attn): Qwen3_5VisionAttention(
            (qkv): Linear(in_features=1024, out_features=3072, bias=True)
            (proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (mlp): Qwen3_5VisionMLP(
            (linear_fc1): Linear(in_features=1024, out_features=4096, bias=True)
            (linear_fc2): Linear(in_features=4096, out_features=1024, bias=True)
            (act_fn): GELUTanh()
         

In [9]:
def push_to_hub(local_dir, repo_name, token):
    """Creates repo and uploads folder to Hugging Face."""
    full_repo_id = f"{HF_USER}/{repo_name}"
    print(f"\n[Hub] Pushing {local_dir} to {full_repo_id}...")

    try:
        api = HfApi()
        create_repo(
            full_repo_id, repo_type="model", exist_ok=True, private=False, token=token
        )

        api.upload_folder(
            folder_path=local_dir, repo_id=full_repo_id, repo_type="model", token=token
        )
        print(f"[Hub] ✅ Successfully uploaded: https://huggingface.co/{full_repo_id}")
    except Exception as e:
        print(f"[Hub] ❌ Error uploading: {e}")

In [10]:
TUNING_CONFIG = {
    "group_size": 32,
    "sym": True,
    "iters": 1000,  # High accuracy (Production grade)
    "nsamples": 512,  # More calibration data
    "batch_size": 1,  # Faster on 48GB VRAM
    "seqlen": 2048*2,
    "low_gpu_mem_usage": False,  # Keep on GPU for speed
    "enable_torch_compile": True,  # JIT acceleration
    "quant_nontext_module": False,  # Keep Vision Tower in FP16 (Crucial for VLM accuracy)
    "layer_config": {
        "mtp": {"data_type": "bfloat16"},
        "mtp.fc": {"data_type": "bfloat16"}
    }
}

In [11]:
ar = AutoRound(
    model=model,
    tokenizer=tokenizer,
    processor=processor,
    scheme="W4A16",
    **TUNING_CONFIG,
)

2026-08-31 10:47:19 INFO entry.py L745: Using MLLM mode for multimodal model.


In [12]:
# SINGLE CALL to save all 3 formats to the same output directory
# The files will exist side-by-side or merged in this folder.
ar.quantize_and_save(
    OUTPUT_BASE_DIR, format="auto_round,auto_gptq,auto_awq,llm_compressor", inplace=True
)

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.
2026-08-31 10:47:22 INFO data_driven.py L772: start to cache block inputs
2026-08-31 10:47:22 INFO mllm.py L83: Using MLLM template: qwen3_5
2026-08-31 10:47:22 INFO calib_dataset.py L977: Preprocessing calibration dataset in a subprocess to avoid memory leaks...
2026-08-31 10:48:34 INFO data_driven.py L795: caching done
Quantizing model.language_model.layers.0:   0%|          | 0/32 [00:07<?, ?it/s]quantized 8/8 layers in the block, loss iter 0: 0.000024 -> iter 757: 0.000001
2026-08-31 10:50:01 INFO device.py L1450: 'peak_ram': 20.25GB, 'peak_vram': 42.74GB
Quantizing model.language_model.layers.1:   3%|▎         | 1/32 [01:26<44:46, 86.68s/it]quantized 8/8 layers in the block, loss iter 0: 0.000020 -> iter 956: 0.000004
2026-08-31 10:50:30 INFO device.py L1450: 'peak_ram': 20.25GB, 'peak_vram': 44.43GB
Quantizing model.language_model.layers.2:   6%|▋         | 2/3

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

2026-08-31 11:04:26 INFO missing_tensors.py L358: Found 15 tensor(s) in the source checkpoint that are absent from the saved output (e.g., MTP parameters): mtp.fc, mtp.layers.0.input_layernorm, mtp.layers.0.mlp.down_proj, mtp.layers.0.mlp.gate_proj, mtp.layers.0.mlp.up_proj, mtp.layers.0.post_attention_layernorm, mtp.layers.0.self_attn.k_norm, mtp.layers.0.self_attn.k_proj, mtp.layers.0.self_attn.o_proj, mtp.layers.0.self_attn.q_norm, mtp.layers.0.self_attn.q_proj, mtp.layers.0.self_attn.v_proj, mtp.norm, mtp.pre_fc_norm_embedding, mtp.pre_fc_norm_hidden. Copying them now...

Loading missing tensors: 100%|██████████| 1/1 [00:00<00:00, 95.33shard/s]
2026-08-31 11:04:26 INFO missing_tensors.py L845: Processing config.json to update quantization_config for missing tensors...
2026-08-31 11:04:26 INFO missing_tensors.py L812: Updated extra_config for 8 ignored layer(s): mtp.fc, mtp.layers.0.mlp.down_proj, mtp.layers.0.mlp.gate_proj, mtp.layers.0.mlp.up_proj, mtp.layers.0.self_attn.k_proj, m

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

2026-08-31 11:05:15 INFO missing_tensors.py L358: Found 15 tensor(s) in the source checkpoint that are absent from the saved output (e.g., MTP parameters): mtp.fc, mtp.layers.0.input_layernorm, mtp.layers.0.mlp.down_proj, mtp.layers.0.mlp.gate_proj, mtp.layers.0.mlp.up_proj, mtp.layers.0.post_attention_layernorm, mtp.layers.0.self_attn.k_norm, mtp.layers.0.self_attn.k_proj, mtp.layers.0.self_attn.o_proj, mtp.layers.0.self_attn.q_norm, mtp.layers.0.self_attn.q_proj, mtp.layers.0.self_attn.v_proj, mtp.norm, mtp.pre_fc_norm_embedding, mtp.pre_fc_norm_hidden. Copying them now...

Loading missing tensors: 100%|██████████| 1/1 [00:00<00:00, 62.00shard/s]
2026-08-31 11:05:16 INFO missing_tensors.py L498: Successfully wrote 15 missing tensor(s) to 'model_extra_tensors.safetensors' in ./NuExtract3/local_model_NuExtract3-w4g32/auto-gptq.
2026-08-31 11:05:16 INFO export.py L167: Saving quantized model to auto_awq format
packing: 100%|██████████| 248/248 [00:25<00:00,  9.57it/s]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

2026-08-31 11:06:06 INFO missing_tensors.py L358: Found 15 tensor(s) in the source checkpoint that are absent from the saved output (e.g., MTP parameters): mtp.fc, mtp.layers.0.input_layernorm, mtp.layers.0.mlp.down_proj, mtp.layers.0.mlp.gate_proj, mtp.layers.0.mlp.up_proj, mtp.layers.0.post_attention_layernorm, mtp.layers.0.self_attn.k_norm, mtp.layers.0.self_attn.k_proj, mtp.layers.0.self_attn.o_proj, mtp.layers.0.self_attn.q_norm, mtp.layers.0.self_attn.q_proj, mtp.layers.0.self_attn.v_proj, mtp.norm, mtp.pre_fc_norm_embedding, mtp.pre_fc_norm_hidden. Copying them now...

Loading missing tensors: 100%|██████████| 1/1 [00:00<00:00, 111.41shard/s]
2026-08-31 11:06:06 INFO missing_tensors.py L498: Successfully wrote 15 missing tensor(s) to 'model_extra_tensors.safetensors' in ./NuExtract3/local_model_NuExtract3-w4g32/auto-awq.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

2026-08-31 11:07:05 INFO missing_tensors.py L358: Found 15 tensor(s) in the source checkpoint that are absent from the saved output (e.g., MTP parameters): mtp.fc, mtp.layers.0.input_layernorm, mtp.layers.0.mlp.down_proj, mtp.layers.0.mlp.gate_proj, mtp.layers.0.mlp.up_proj, mtp.layers.0.post_attention_layernorm, mtp.layers.0.self_attn.k_norm, mtp.layers.0.self_attn.k_proj, mtp.layers.0.self_attn.o_proj, mtp.layers.0.self_attn.q_norm, mtp.layers.0.self_attn.q_proj, mtp.layers.0.self_attn.v_proj, mtp.norm, mtp.pre_fc_norm_embedding, mtp.pre_fc_norm_hidden. Copying them now...

Loading missing tensors: 100%|██████████| 1/1 [00:00<00:00, 102.66shard/s]
2026-08-31 11:07:06 INFO missing_tensors.py L498: Successfully wrote 15 missing tensor(s) to 'model_extra_tensors.safetensors' in ./NuExtract3/local_model_NuExtract3-w4g32/llm-compressor-wint-a16.
2026-08-31 11:07:06 INFO device.py L1450: 'peak_ram': 20.25GB, 'peak_vram': 48.62GB


(Qwen3_5ForConditionalGeneration(
   (model): Qwen3_5Model(
     (visual): Qwen3_5VisionModel(
       (patch_embed): Qwen3_5VisionPatchEmbed(
         (proj): Conv3d(3, 1024, kernel_size=(2, 16, 16), stride=(2, 16, 16))
       )
       (pos_embed): Embedding(2304, 1024)
       (rotary_pos_emb): Qwen3_5VisionRotaryEmbedding()
       (blocks): ModuleList(
         (0-23): 24 x Qwen3_5VisionBlock(
           (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True, bias=True)
           (norm2): LayerNorm((1024,), eps=1e-06, elementwise_affine=True, bias=True)
           (attn): Qwen3_5VisionAttention(
             (qkv): Linear(in_features=1024, out_features=3072, bias=True)
             (proj): Linear(in_features=1024, out_features=1024, bias=True)
           )
           (mlp): Qwen3_5VisionMLP(
             (linear_fc1): Linear(in_features=1024, out_features=4096, bias=True)
             (linear_fc2): Linear(in_features=4096, out_features=1024, bias=True)
             (act_fn): 

In [13]:
notebook_login()

In [14]:
HF_USER = "Vishva007"

In [15]:
base_name = MODEL_ID.split("/")[-1]
hf_token = get_token()

In [16]:
if hf_token:
    push_to_hub(
        os.path.join(OUTPUT_BASE_DIR, "local_model_NuExtract3-w4g32/auto-round-auto-gptq"), 
        f"{base_name}-W4A16-AutoRound", 
        hf_token)
    push_to_hub(
        os.path.join(OUTPUT_BASE_DIR, "local_model_NuExtract3-w4g32/auto-gptq"), 
        f"{base_name}-W4A16-AutoRound-GPTQ",
        hf_token
    )
    push_to_hub(
        os.path.join(OUTPUT_BASE_DIR, "local_model_NuExtract3-w4g32/llm-compressor-wint-a16"), 
        f"{base_name}-W4A16-AutoRound-LLM-Compressor", 
        hf_token
    )
else:
    print("No Hugging Face token found. Skipping upload to hub.")


[Hub] Pushing ./NuExtract3/local_model_NuExtract3-w4g32/auto-round-auto-gptq to Vishva007/NuExtract3-W4A16-AutoRound...
[Hub] ✅ Successfully uploaded: https://huggingface.co/Vishva007/NuExtract3-W4A16-AutoRound

[Hub] Pushing ./NuExtract3/local_model_NuExtract3-w4g32/auto-gptq to Vishva007/NuExtract3-W4A16-AutoRound-GPTQ...
[Hub] ✅ Successfully uploaded: https://huggingface.co/Vishva007/NuExtract3-W4A16-AutoRound-GPTQ

[Hub] Pushing ./NuExtract3/local_model_NuExtract3-w4g32/llm-compressor-wint-a16 to Vishva007/NuExtract3-W4A16-AutoRound-LLM-Compressor...
[Hub] ✅ Successfully uploaded: https://huggingface.co/Vishva007/NuExtract3-W4A16-AutoRound-LLM-Compressor
